# Bayesian Classification Model

In [1]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

## Metricas de Bayesian Classification Model

In [ ]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------

import numpy as np # type: ignore
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from joblib import dump
from time import time
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical
from sklearn.naive_bayes import GaussianNB, Real
from skopt.space import Real

# ------------------------
# Paso 2: Cargar los datos
# ------------------------

y_train = Data_final['isFraud']
x_train = Data_final.drop(columns=['isFraud']) # anexar base de datos de JESÚS.
x_train, x_test, y_train, y_test = train_test_split(x_train,y_train,test_size=0.20,random_state=90,stratify=y_train)

# ------------------------
# Paso 3: Definir el pipeline
# ------------------------
pipe_nb = Pipeline([
    ('scaler', StandardScaler()),  # En algunos casos NB no requiere escalado, pero lo incluimos como opción
    ('nb', GaussianNB())
])

# ------------------------
# Paso 4: Definir espacio de búsqueda para BayesSearchCV
# ------------------------
search_spaces = {
    'nb__var_smoothing': Real(1e-11, 1e-7, prior='log-uniform')  # hiperparámetro sensible en GaussianNB
}

# ------------------------
# Paso 5: Entrenar el modelo con BayesSearchCV
# ------------------------
bayes_nb = BayesSearchCV(
    estimator=pipe_nb,
    search_spaces=search_spaces,
    n_iter=20,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_nb.fit(x_train, y_train)
training_time_nb = time() - start_time

# Guardar el modelo
dump(bayes_nb, 'bayes_nb.joblib')

# ------------------------
# Paso 6: Hacer predicciones
# ------------------------
y_pred_nb = bayes_nb.best_estimator_.predict(x_test)
y_pred_proba_nb = bayes_nb.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Calcular métricas
# ------------------------
precision_nb = precision_score(y_test, y_pred_nb, average='weighted')
recall_nb = recall_score(y_test, y_pred_nb, average='weighted')
accuracy_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average='weighted')
auc_nb = roc_auc_score(y_test, y_pred_proba_nb)

# ------------------------
# Paso 8: Resultados en DataFrame
# ------------------------
resultados_nb = pd.DataFrame({
    'Precision': [f"{precision_nb:.2f}"],
    'Recall': [f"{recall_nb:.2f}"],
    'Accuracy': [f"{accuracy_nb:.2f}"],
    'F1-Score': [f"{f1_nb:.2f}"],
    'AUC': [f"{auc_nb:.2f}"],
    'CPU time (s)': [round(training_time_nb, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo Naive Bayes (Bayesian Optimization):")
display(resultados_nb)

In [3]:
display(resultados_nb)

,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.95,0.22,0.22,0.32,0.71,46.95


El modelo bayesiano muestra un rendimiento significativamente inferior comparado con los modelos anteriores. La precisión del 95% es alta, lo que significa que cuando el modelo predice una transacción como fraudulenta, esa predicción es generalmente correcta. Sin embargo, el recall extremadamente bajo del 22% indica que el modelo está perdiendo la mayoría de las transacciones fraudulentas reales. La accuracy del 22% confirma la debilidad del modelo en la clasificación general. El F1-Score de 0.32 refleja el desequilibrio crítico entre precisión y recall, sugiriendo una capacidad muy limitada para detectar fraudes. El AUC de 0.71, aunque aceptable, no compensa los pobres resultados en otras métricas. El tiempo de CPU de solo 46.95 segundos, significativamente menor que los otros modelos, podría indicar una menor complejidad computacional, pero no compensa su baja efectividad en la detección de fraudes.